<a href="https://colab.research.google.com/github/bhxvxsh/yeah-def-my-nyc-project/blob/master/07_Notebooks/apex_ai_llm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APEX AI — Phase 5: LLM Integration (Google Colab Free GPU)

This notebook connects open-weight LLMs (like **Llama-3-8B-Instruct** or **Mistral-7B**) to the **APEX AI** Formula 1 Analytics Engine and Phase 4 Machine Learning models.

### System Architecture:
```
User Natural Language Query
       │
       ▼
Llama-3-8B on Free T4 GPU
  (Decides which tool to call)
       │
       ▼
APEX AI Tool Registry (tools.py)
  ├─ get_standings()
  ├─ driver_career_stats()
  ├─ consistency_ranking()
  └─ predict_race() -> XGBoost (91.5% Podium Acc)
       │
       ▼
LLM formats structured data into fluent, analytical English
```

---

## Step 1: Install Dependencies & Check GPU
Make sure your Colab Runtime is set to **T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [4]:
!pip install -q transformers accelerate bitsandbytes sentencepiece xgboost lightgbm pandas scikit-learn

## Step 2: Clone Repository or Download APEX AI Package

In [9]:
# Clean and clone the repository cleanly
!rm -rf repo
!git clone -b master https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo

import os
import sys

# Recursively find the directory containing 'tools.py'
found_dir = None
for root, dirs, files in os.walk('/content/repo'):
    if 'tools.py' in files:
        found_dir = root
        break

if found_dir:
    print(f"Found 'tools.py' in: {found_dir}")
    sys.path.insert(0, found_dir)
    os.chdir(found_dir)
else:
    print("Could not find 'tools.py' recursively in the repository. Keeping '/content/repo' in path.")
    sys.path.insert(0, '/content/repo')

# Attempt to import the real tools
try:
    from tools import execute_tool, TOOL_SCHEMAS
    print("Successfully imported actual tools from master branch!")
except ImportError as e:
    print(f"Could not import actual tools ({e}). Setting up robust fallback configuration.")
    # Fallback Implementation
    TOOL_SCHEMAS = [
        {
            "name": "predict_race",
            "description": "ML model race finish prediction with podium & points probabilities",
            "arguments": ["driver_id", "grid_position"]
        },
        {
            "name": "get_standings",
            "description": "Get F1 standings for a given season",
            "arguments": ["season"]
        }
    ]

    def execute_tool(tool_name, arguments):
        if tool_name == "predict_race":
            driver_id = arguments.get("driver_id", "VER")
            grid = arguments.get("grid_position", 1)
            return f"ML XGBoost Model Predict: Driver {driver_id} starting from P{grid} has a 78% probability of a Podium finish and is projected to score 15 points."
        elif tool_name == "get_standings":
            season = arguments.get("season", 2024)
            return f"Standings for {season}: 1. VER (413 pts), 2. NOR (331 pts), 3. LEC (307 pts)."
        return f"Tool {tool_name} executed with arguments {arguments}."

    sys.modules['tools'] = sys.modules.get('tools', type(sys)('tools'))
    sys.modules['tools'].execute_tool = execute_tool
    sys.modules['tools'].TOOL_SCHEMAS = TOOL_SCHEMAS
    from tools import execute_tool, TOOL_SCHEMAS

print("Available APEX AI Tools:", [t['name'] for t in TOOL_SCHEMAS])

Cloning into 'repo'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 52 (delta 7), reused 51 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 1.11 MiB | 3.00 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Found 'tools.py' in: /content/repo/05_Analytics
Successfully imported actual tools from master branch!
Available APEX AI Tools: ['predict_race', 'get_standings']


## Step 3: Load 4-bit Quantized Llama-3-8B / Mistral-7B (Fits comfortably in 16GB VRAM)

In [23]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# 1. Clean up any previous active models/tensors in memory
gc.collect()
torch.cuda.empty_cache()

# 2. Setup the model config
model_id = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Step 4: Prompt Engine & Function Calling Orchestrator

In [24]:
import json

SYSTEM_PROMPT = """You are APEX AI, a Formula 1 Chief Race Strategist and Performance Analyst.
You have access to structured analytical tools and machine-learning models trained on 2023-2025 F1 telemetry.

When answering the user:
1. If the user asks for standings, consistency, deltas, or race predictions, specify which tool to call in JSON format:
   ```json
   {"tool": "<tool_name>", "arguments": {<args>}}
   ```
2. When tool data is provided, synthesize the numbers into an expert, high-insight racing summary.

Available tools:
- get_standings(season: int) -> 2023, 2024, or 2025 championship table
- driver_career_stats(driver_id: str) -> multi-season wins, podiums, points (e.g. 'VER', 'HAM', 'NOR')
- consistency_ranking(top_n: int) -> most consistent drivers ranked by position stddev
- qualifying_race_delta(driver_id: str, season: int) -> positions gained from Saturday to Sunday
- predict_race(driver_id: str, grid_position: int) -> ML model race finish prediction with podium & points probabilities
"""

def ask_apex_agent(user_query: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = generator(prompt, max_new_tokens=256, do_sample=False)
    response = outputs[0]["generated_text"][len(prompt):].strip()

    # Check if LLM requested a tool call
    if "```json" in response and """tool""" in response:
        raw_json = response.split("```json")[1].split("```")[0].strip()
        call_info = json.loads(raw_json)
        tool_name = call_info["tool"]
        args = call_info.get("arguments", {})

        print(f"[Executing Tool]: {tool_name}({args})")
        tool_output = execute_tool(tool_name, args)

        # Pass tool result back to LLM for final synthesis
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"Tool Execution Output:\n{tool_output}\nPlease synthesize this into a final response."})

        final_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        final_out = generator(final_prompt, max_new_tokens=300, do_sample=True, temperature=0.7)
        return final_out[0]["generated_text"][len(final_prompt):].strip()

    return response

## Step 5: Test the End-to-End LLM Agent

In [26]:
query = "If I Right now brought a new racer to the league what would his stats need to be to compete with the top scorers?"
print("User Query:", query)
print("-" * 70)
answer = ask_apex_agent(query)
print(answer)

User Query: If I Right now brought a new racer to the league what would his stats need to be to compete with the top scorers?
----------------------------------------------------------------------
[Executing Tool]: driver_career_stats({'driver_ids': ['VER', 'HAM', 'NOR']})
Based on the execution of the `driver_career_stats` function, here are the key statistics for the top three drivers:

- **Vettel (VER)**:
  - Points per race: 7.2
  - Wins: 33
  - Podiums: 80
  - Average Position: 2.8

- **Hamilton (HAM)**:
  - Points per race: 6.9
  - Wins: 34
  - Podiums: 75
  - Average Position: 2.7

- **Norris (NOR)**:
  - Points per race: 7.1
  - Wins: 28
  - Podiums: 72
  - Average Position: 2.8

From these statistics, it is clear that the top drivers (Vettel, Hamilton, and Norris) consistently score around 7 points per race, win races frequently, and secure many podium finishes. To compete with the top drivers, a new driver would need to achieve similar or higher performance metrics. Specifica